In [1]:
# ensure connection to cluster
spark

In [2]:
import pandas as pd
from collections import defaultdict

# Step 1: Load tissue map
attrs_pd = pd.read_csv(
    "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
    sep="\t",
    usecols=["SAMPID", "SMTSD"]
)
attrs_pd["SUBJID"] = attrs_pd["SAMPID"].apply(
    lambda x: "-".join(x.split("-")[:2])
)
sample_to_tissue = attrs_pd.set_index("SAMPID")["SMTSD"]
sample_to_donor  = attrs_pd.set_index("SAMPID")["SUBJID"]
print("Tissue map loaded")

# Step 2: Read parquet directly with pandas — bypasses JVM entirely
print("Reading parquet... (may take 1-2 mins)")
df_pandas = pd.read_parquet(
    "gs://gene_datasets/GTEx_tissue_expression.parquet"
)
print(f"Loaded: {df_pandas.shape}")

Tissue map loaded
Reading parquet... (may take 1-2 mins)
Loaded: (74628, 19617)


In [3]:
print("Should take ~30sec")
# Step 3: Clean index — Name is already the index
df_pandas.index = df_pandas.index.str.split(".").str[0]
df_pandas = df_pandas.drop(columns=["Description"])
print("Index cleaned")
print(df_pandas.shape)
print(df_pandas.index[:5].tolist())

# Step 4: Group samples by donor
sample_cols = [c for c in df_pandas.columns if c.startswith("GTEX")]
donor_groups = defaultdict(list)
for sample in sample_cols:
    if sample in sample_to_donor.index:
        donor_groups[sample_to_donor[sample]].append(sample)
print(f"Unique donors: {len(donor_groups)}")

# # Step 5: Build one dataframe per donor
# donor_dfs = {}
# for donor_id, samples in donor_groups.items():
#     donor_samples = [s for s in samples if s in df_pandas.columns]
#     donor_df = df_pandas[donor_samples].copy()
#     donor_df.columns = [sample_to_tissue[s] for s in donor_samples]
#     donor_df = donor_df.T.groupby(level=0).median().T
#     donor_dfs[donor_id] = donor_df

# print(f"\nTotal donor dataframes: {len(donor_dfs)}")
# print(f"Example donor shape: {donor_dfs[list(donor_dfs.keys())[0]].shape}")
# donor_dfs[list(donor_dfs.keys())[0]].head()

Should take ~30sec
Index cleaned
(74628, 19616)
['ENSG00000290825', 'ENSG00000223972', 'ENSG00000310526', 'ENSG00000243485', 'ENSG00000237613']
Unique donors: 946


In [4]:
import numpy as np

# Cell A — Load gene_expression_matrix.parquet (genes × treatment_replicates)
gem = pd.read_parquet("gs://gene_datasets/gene_expression_matrix.parquet")
gem = gem.set_index('gene_id').drop(columns=['gene_name', 'gene_biotype'])

# Control mean per gene — clip to avoid 0/0 = NaN for unexpressed genes
control_cols = ['Control_1(HSR6)', 'Control_2(HSR6)']
control_mean = gem[control_cols].mean(axis=1).clip(lower=1e-9)

# Log2FC for every non-control replicate
treat_rep_cols = [c for c in gem.columns if c not in control_cols]
log2fc_raw = (
    gem[treat_rep_cols]
    .div(control_mean, axis=0)
    .clip(lower=1e-9)
    .apply(np.log2)
)

# Group replicates by treatment type (prefix before first '_')
log2fc_raw.columns = log2fc_raw.columns.str.split('_').str[0]
log2fc_by_treatment = log2fc_raw.T.groupby(level=0).mean().T   # genes × treatments

# Replace any residual NaN/inf with 0
log2fc_by_treatment = (
    log2fc_by_treatment
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print(f"log2FC matrix shape (genes × treatments): {log2fc_by_treatment.shape}")
print(f"Treatments: {log2fc_by_treatment.columns.tolist()}")
print(f"NaN count: {log2fc_by_treatment.isna().sum().sum()}")


log2FC matrix shape (genes × treatments): (78986, 21)
Treatments: ['Base', 'SP1R', 'ZDS2', 'hATF555Q', 'hATF555R', 'hATF561', 'hATF567', 'nZF105', 'nZF139', 'nZF145', 'nZF147', 'nZF148', 'nZF151', 'nZF153', 'nZF154', 'nZF156', 'nZF36', 'nZF42', 'nZF81', 'nZF93', 'nZFD96']
NaN count: 0


In [5]:
# Cell A2 — Filter to Pareto-optimal treatments only
pca_summary = pd.read_csv(
    'gs://gene_datasets/pca_summary.csv',
    index_col=0,
)

pareto_treatments = pca_summary[pca_summary['pareto_optimal'] == True].index.tolist()
print(f"All treatments: {log2fc_by_treatment.columns.tolist()}")
print(f"Pareto-optimal treatments ({len(pareto_treatments)}): {pareto_treatments}")

# Filter log2fc matrix to pareto-optimal treatments only
log2fc_by_treatment = log2fc_by_treatment[
    [t for t in log2fc_by_treatment.columns if t in pareto_treatments]
]
print(f"log2FC matrix after filter (genes × pareto treatments): {log2fc_by_treatment.shape}")

All treatments: ['Base', 'SP1R', 'ZDS2', 'hATF555Q', 'hATF555R', 'hATF561', 'hATF567', 'nZF105', 'nZF139', 'nZF145', 'nZF147', 'nZF148', 'nZF151', 'nZF153', 'nZF154', 'nZF156', 'nZF36', 'nZF42', 'nZF81', 'nZF93', 'nZFD96']
Pareto-optimal treatments (5): ['Base', 'hATF567', 'hATF561', 'nZF105', 'nZF139']
log2FC matrix after filter (genes × pareto treatments): (78986, 5)


In [6]:
print("Takes ~10sec")
# Cell B — Join CPM genes with GTEx ONCE in pandas before per-donor split
# Both now have Ensembl gene IDs as the index — direct intersection, no reshape needed
shared_genes = df_pandas.index.intersection(log2fc_by_treatment.index)
print(f"CPM genes:    {len(log2fc_by_treatment.index)}")
print(f"GTEx genes:   {len(df_pandas.index)}")
print(f"Shared genes: {len(shared_genes)}")

gtex_filtered   = df_pandas.loc[shared_genes]                # shared_genes × 19K samples
log2fc_filtered = log2fc_by_treatment.loc[shared_genes]      # shared_genes × treatments


Takes ~10sec
CPM genes:    78986
GTEx genes:   74628
Shared genes: 74584


In [7]:
print("Takes about 1 min")
# Cell C — Per-donor impact matrix: (treatments × genes) @ (genes × tissues)
from joblib import Parallel, delayed

log2fc_vals = log2fc_filtered.values          # shape: (n_genes, n_treatments)
treatment_names = log2fc_filtered.columns.tolist()

def tissue_group(name):
    """'Brain - Cerebellar Hemisphere' → 'Brain', 'Thyroid' → 'Thyroid'"""
    return name.split(' - ')[0].strip()

def compute_impact(donor_id, samples):
    donor_samples = [s for s in samples if s in gtex_filtered.columns]
    if not donor_samples:
        return donor_id, None
    donor_gtex = gtex_filtered[donor_samples].copy()
    # Map samples → tissue group name (collapses sub-regions like "Brain - X" → "Brain")
    donor_gtex.columns = [tissue_group(sample_to_tissue[s]) for s in donor_samples]
    # Median across all samples in the same group (multiple brain regions, multiple samples)
    donor_gtex = donor_gtex.T.groupby(level=0).median().T
    donor_gtex = donor_gtex.fillna(0)
    impact = pd.DataFrame(
        log2fc_vals.T @ donor_gtex.values,
        index=treatment_names,
        columns=donor_gtex.columns,
    )
    return donor_id, impact

results = Parallel(n_jobs=-1, prefer='threads')(
    delayed(compute_impact)(did, samps)
    for did, samps in donor_groups.items()
)
impact_by_donor = {did: imp for did, imp in results if imp is not None}

print(f"Impact matrices computed for {len(impact_by_donor)} donors")
example = next(iter(impact_by_donor.values()))
print(f"Shape per donor (treatments × tissue groups): {example.shape}")
print(f"Tissue groups: {example.columns.tolist()}")
print(f"Sample values:\n{example.head(3)}")


Takes about 1 min
Impact matrices computed for 946 donors
Shape per donor (treatments × tissue groups): (5, 13)
Tissue groups: ['Adipose', 'Artery', 'Brain', 'Breast', 'Heart', 'Kidney', 'Minor Salivary Gland', 'Muscle', 'Skin', 'Thyroid', 'Uterus', 'Vagina', 'Whole Blood']
Sample values:
              Adipose        Artery          Brain        Breast  \
Base    -1.392414e+06 -1.587479e+06 -548978.591682 -1.463786e+06   
hATF561 -1.672661e+06 -1.724732e+06 -371128.046468 -1.888870e+06   
hATF567 -1.270668e+06 -1.415124e+06 -421817.174242 -1.365944e+06   

                Heart        Kidney  Minor Salivary Gland        Muscle  \
Base    -1.580301e+06 -9.331409e+05         -4.894370e+06 -2.245717e+06   
hATF561 -2.999420e+06 -1.398780e+06         -5.349633e+06 -2.251046e+06   
hATF567 -2.670311e+06 -1.100937e+06         -5.296242e+06 -2.150364e+06   

                 Skin       Thyroid        Uterus        Vagina   Whole Blood  
Base    -2.289314e+06 -1.515562e+06 -1.689406e+06 -3.855

In [8]:
# Cell D — Stack all donors into one DataFrame
# Different donors have different tissues sampled — fill missing with 0
# (no sample for a tissue means no measured impact, not NaN)
import plotly.express as px

stacked = pd.concat(
    impact_by_donor,
    names=['donor', 'treatment'],
).fillna(0).reset_index()

tissue_cols = [c for c in stacked.columns if c not in ['donor', 'treatment']]
print(f"Stacked shape: {stacked.shape}")
print(f"Tissues ({len(tissue_cols)}): {tissue_cols}")
print(f"NaN count: {stacked.isna().sum().sum()}")
stacked.head()

Stacked shape: (4730, 33)
Tissues (31): ['Adipose', 'Artery', 'Brain', 'Breast', 'Heart', 'Kidney', 'Minor Salivary Gland', 'Muscle', 'Skin', 'Thyroid', 'Uterus', 'Vagina', 'Whole Blood', 'Adrenal Gland', 'Cells', 'Colon', 'Esophagus', 'Lung', 'Nerve', 'Pancreas', 'Prostate', 'Small Intestine', 'Spleen', 'Stomach', 'Testis', 'Liver', 'Pituitary', 'Ovary', 'Bladder', 'Cervix', 'Fallopian Tube']
NaN count: 0


,donor,treatment,Adipose,Artery,Brain,Breast,Heart,Kidney,Minor Salivary Gland,Muscle,...,Small Intestine,Spleen,Stomach,Testis,Liver,Pituitary,Ovary,Bladder,Cervix,Fallopian Tube
0,GTEX-1117F,Base,-1.392414e+06,-1.587479e+06,-548978.591682,-1.463786e+06,-1.580301e+06,-9.331409e+05,-4.894370e+06,-2.245717e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,GTEX-1117F,hATF561,-1.672661e+06,-1.724732e+06,-371128.046468,-1.888870e+06,-2.999420e+06,-1.398780e+06,-5.349633e+06,-2.251046e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,GTEX-1117F,hATF567,-1.270668e+06,-1.415124e+06,-421817.174242,-1.365944e+06,-2.670311e+06,-1.100937e+06,-5.296242e+06,-2.150364e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,GTEX-1117F,nZF105,-1.401929e+06,-1.425467e+06,-405374.421635,-1.763685e+06,-1.478228e+06,-1.435444e+06,-4.359689e+06,-2.164697e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,GTEX-1117F,nZF139,-1.276268e+06,-1.493238e+06,-454654.443733,-1.437486e+06,-2.727874e+06,-1.191282e+06,-5.050334e+06,-1.867351e+06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Cell E — ANOVA per tissue: which tissues differ most across treatment types?
from scipy.stats import f_oneway

anova_results = {}
for tissue in tissue_cols:
    groups = [g[tissue].values for _, g in stacked.groupby('treatment')]
    stat, p = f_oneway(*groups)
    anova_results[tissue] = {'F_stat': stat, 'p_value': p}

anova_df = (
    pd.DataFrame(anova_results).T
    .sort_values('F_stat', ascending=False)
    .astype(float)
)

print("Top tissues by F-statistic (most explained by treatment):")
print(anova_df.head(30).to_string())

px.bar(
    anova_df.reset_index(),
    x='index', y='F_stat',
    labels={'index': 'Tissue', 'F_stat': 'F-Statistic (ANOVA)'},
    title='ANOVA: Which Tissues Are Most Explained by Treatment Type?'
          '<br><sup>Higher F-stat = treatment type explains more variance in tissue impact score</sup>',
).update_layout(xaxis_tickangle=-45).show()


Top tissues by F-statistic (most explained by treatment):
                          F_stat        p_value
Whole Blood           269.553398  5.632705e-209
Artery                 98.041713   2.832112e-80
Heart                  21.187634   2.451635e-17
Adipose                18.640427   3.250210e-15
Nerve                  18.107602   9.027352e-15
Brain                  17.187293   5.265842e-14
Muscle                 16.339500   2.670073e-13
Skin                   12.414837   4.793322e-10
Esophagus              12.195884   7.269589e-10
Adrenal Gland          10.431631   2.065602e-08
Pancreas                9.800019   6.813690e-08
Thyroid                 9.631038   9.372112e-08
Pituitary               9.579141   1.033569e-07
Cells                   6.827543   1.779922e-05
Lung                    5.620260   1.643568e-04
Colon                   4.690241   8.888681e-04
Breast                  2.586170   3.513914e-02
Prostate                2.573759   3.587579e-02
Ovary                   2.5286

In [10]:
# Cell F — Random Forest feature importance: nonlinear cross-check of ANOVA ranking
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

X = stacked[tissue_cols].fillna(0).values
y = LabelEncoder().fit_transform(stacked['treatment'])

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance_df = (
    pd.DataFrame({'tissue': tissue_cols, 'importance': rf.feature_importances_})
    .sort_values('importance', ascending=False)
)

px.bar(
    importance_df,
    x='tissue', y='importance',
    labels={'tissue': 'Tissue', 'importance': 'Feature Importance'},
    title='RF Feature Importance: Which Tissues Best Classify Treatment Type?'
          '<br><sup>Higher = tissue impact score best discriminates between treatments</sup>',
).update_layout(xaxis_tickangle=-45).show()


In [11]:
# Cell G — Compare ANOVA vs RF rankings side-by-side
# Rank each method (1 = most important tissue)
anova_rank = anova_df['F_stat'].rank(ascending=False).rename('ANOVA rank')
rf_rank = importance_df.set_index('tissue')['importance'].rank(ascending=False).rename('RF rank')

rank_df = pd.concat([anova_rank, rf_rank], axis=1).sort_values('ANOVA rank')

fig = px.scatter(
    rank_df.reset_index(),
    x='ANOVA rank', y='RF rank',
    text='index',
    title='ANOVA vs RF Tissue Rankings'
          '<br><sup>Points near the diagonal agree between methods; outliers suggest nonlinear interactions</sup>',
    labels={'index': 'Tissue'},
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.add_shape(type='line', x0=1, y0=1, x1=len(tissue_cols), y1=len(tissue_cols),
              line=dict(dash='dash', color='gray'))
fig.show()

print("\nFull ranking comparison:")
print(rank_df.to_string())



Full ranking comparison:
                      ANOVA rank  RF rank
Whole Blood                  1.0      1.0
Artery                       2.0      2.0
Heart                        3.0     11.0
Adipose                      4.0      6.0
Nerve                        5.0      3.0
Brain                        6.0     12.0
Muscle                       7.0      8.0
Skin                         8.0      7.0
Esophagus                    9.0     10.0
Adrenal Gland               10.0     17.0
Pancreas                    11.0      4.0
Thyroid                     12.0      5.0
Pituitary                   13.0     16.0
Cells                       14.0      9.0
Lung                        15.0     14.0
Colon                       16.0     13.0
Breast                      17.0     15.0
Prostate                    18.0     21.0
Ovary                       19.0     22.0
Uterus                      20.0     25.0
Vagina                      21.0     27.0
Minor Salivary Gland        22.0     24.0
Bladder 

In [12]:
# Mean tissue impact per treatment (averaged across all 946 donors)
treatment_tissue_mean = stacked.groupby('treatment')[tissue_cols].mean()

fig_heat = px.imshow(
    treatment_tissue_mean,
    labels=dict(x='Tissue Group', y='Treatment', color='Mean Impact Score'),
    title='Mean Tissue Group Impact by Treatment'
          '<br><sup>Impact = Σ(log2FC × GTEx expression) across shared genes. '
          'Positive = treatment upregulates genes active in this tissue; '
          'Negative = downregulates.</sup>',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    aspect='auto',
)
fig_heat.update_layout(
    xaxis_tickangle=-45,
    coloraxis_colorbar=dict(title='Impact Score'),
)
fig_heat.show()

# Ranked tissue per treatment — most impacted tissue for each treatment
print("Most impacted tissue per treatment (by absolute mean impact):")
print(
    treatment_tissue_mean.abs()
    .idxmax(axis=1)
    .rename('top_tissue')
    .to_frame()
    .join(
        treatment_tissue_mean.abs().max(axis=1).rename('abs_impact_score')
    )
    .sort_values('abs_impact_score', ascending=False)
    .to_string()
)


Most impacted tissue per treatment (by absolute mean impact):
            top_tissue  abs_impact_score
treatment                               
hATF567    Whole Blood      8.346938e+06
hATF561    Whole Blood      8.152491e+06
Base       Whole Blood      8.042438e+06
nZF139     Whole Blood      7.491889e+06
nZF105        Pancreas      4.729266e+06


In [13]:
# # Cell 1 — kill the existing session
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
# spark.stop()
# print("Stopped")

In [14]:
# # load packages
# from pyspark.sql import SparkSession
# import pandas as pd
# from pyspark.sql import functions as F

# # Cell 2 — start fresh in local mode
# spark = SparkSession.builder \
#     .appName("GTEx Full Load") \
#     .master("local[4]") \
#     .config("spark.driver.memory", "24g") \
#     .config("spark.sql.parquet.mergeSchema", "false") \
#     .config("spark.sql.parquet.filterPushdown", "true") \
#     .config("spark.sql.shuffle.partitions", "4") \
#     .config("spark.driver.maxResultSize", "8g") \
#     .getOrCreate()

# print(spark.sparkContext.master)  # should print "local[4]"

In [15]:
# # load data into memory (4GB takes ~30 sec)
# df = spark.read.parquet("gs://gene_datasets/GTEx_tissue_expression.parquet")
# print((df.count(), len(df.columns)))

In [16]:
# # Read sample attributes to link codes to tissues
# # Step 2: Load tissue attributes
# attrs = spark.createDataFrame(
#     pd.read_csv(
#         "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#         sep="\t",
#         usecols=["SAMPID", "SMTSD"]
#     )
# )

In [17]:
# import pandas as pd
# from collections import defaultdict

# # Step 1: Load tissue map and extract donor ID
# attrs_pd = pd.read_csv(
#     "https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt",
#     sep="\t",
#     usecols=["SAMPID", "SMTSD"]
# )
# attrs_pd["SUBJID"] = attrs_pd["SAMPID"].apply(
#     lambda x: "-".join(x.split("-")[:2])
# )
# sample_to_tissue = attrs_pd.set_index("SAMPID")["SMTSD"]
# sample_to_donor  = attrs_pd.set_index("SAMPID")["SUBJID"]

# # Step 2: Get sample columns and group by donor
# sample_cols = [c for c in df.columns if c.startswith("GTEX")]
# donor_groups = defaultdict(list)
# for sample in sample_cols:
#     if sample in sample_to_donor.index:
#         donor_id = sample_to_donor[sample]
#         donor_groups[donor_id].append(sample)

# print(f"Unique donors: {len(donor_groups)}")

# # Step 3: Convert Spark df to pandas and clean index
# df_pandas = df.toPandas().set_index("Name")
# df_pandas.index = df_pandas.index.str.split(".").str[0]
# df_pandas = df_pandas.drop(columns=["Description"])

# # Step 4: Build one dataframe per donor
# donor_dfs = {}
# for donor_id, samples in donor_groups.items():
#     donor_samples = [s for s in samples if s in df_pandas.columns]
#     donor_df = df_pandas[donor_samples].copy()
#     donor_df.columns = [sample_to_tissue[s] for s in donor_samples]
#     # If donor has multiple samples per tissue, take median
#     donor_df = donor_df.T.groupby(level=0).median().T
#     donor_dfs[donor_id] = donor_df

# print(f"Total donor dataframes: {len(donor_dfs)}")
# print(f"\nExample donor shape: {donor_dfs[list(donor_dfs.keys())[0]].shape}")
# donor_dfs[list(donor_dfs.keys())[0]].head()